In [1]:
import numpy as np
import pandas as pd

# Dataset & DataLoader

In [2]:
import torch
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T

from dataset import TrainDataset, TestDataset

image_size = 64
batch_size = 64
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_transform = T.Compose([
    T.RandomResizedCrop(image_size, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_dataset = TrainDataset(root_path = './cs441-assn3-data/Train_64/', transform = train_transform)
test_dataset = TestDataset(root_path = './cs441-assn3-data/Test_64/', transform = eval_transform)

val_ratio = 0.2 # train 80%, val 20%

# 전체 길이 기준으로 train/val 길이 계산
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

train_dataset_split, val_dataset = random_split(
    train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(dataset=train_dataset_split,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last = True,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

# Your Awesome Model

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


2.8.0+cu129
True


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LayerNorm(nn.Module):
    """
    ConvNeXt에서 사용하는 LayerNorm.
    채널 차원이 마지막에 오는 것(data_format='channels_last')과
    앞에 오는 것(data_format='channels_first') 모두 지원
    """
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        if self.data_format not in ["channels_last", "channels_first"]:
            raise NotImplementedError 
        self.normalized_shape = (normalized_shape, )
    
    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None] * x + self.bias[:, None, None]
            return x

class Block(nn.Module):
    """
    ConvNeXt Block
    """
    def __init__(self, dim, drop_path=0.):
        super().__init__()
        # Depthwise Conv (7x7)
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim) 
        self.norm = LayerNorm(dim, eps=1e-6)
        
        # Pointwise Conv (1x1) implemented with Linear layers
        self.pwconv1 = nn.Linear(dim, 4 * dim) 
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        
        self.gamma = nn.Parameter(1e-6 * torch.ones((dim)), requires_grad=True) if dim > 0 else None
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1) # (N, C, H, W) -> (N, H, W, C)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        
        if self.gamma is not None:
            x = self.gamma * x
            
        x = x.permute(0, 3, 1, 2) # (N, H, W, C) -> (N, C, H, W)
        x = input + self.drop_path(x)
        return x

class DropPath(nn.Module):
    """
    Stochastic Depth (ResNet 등 깊은 모델 학습 시 필수적인 정규화)
    """
    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor

class ConvNeXt(nn.Module):
    def __init__(self, in_chans=3, num_classes=15, 
                 depths=[3, 3, 27, 3], dims=[96, 192, 384, 768], drop_path_rate=0.4): # Small config
        super().__init__()
        
        # 1. Stem (이미지 초반 처리)
        # 64x64 이미지 손실을 줄이기 위해 stride=1 사용 (원래는 4)
        self.downsample_layers = nn.ModuleList() 
        stem = nn.Sequential(
            nn.Conv2d(in_chans, dims[0], kernel_size=3, stride=1, padding=1),
            LayerNorm(dims[0], eps=1e-6, data_format="channels_first")
        )
        self.downsample_layers.append(stem)
        
        # 2. Downsample Layers (Stage 사이)
        for i in range(3):
            downsample_layer = nn.Sequential(
                    LayerNorm(dims[i], eps=1e-6, data_format="channels_first"),
                    nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2),
            )
            self.downsample_layers.append(downsample_layer)

        # 3. Stages (Blocks)
        self.stages = nn.ModuleList() 
        dp_rates = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))] 
        cur = 0
        for i in range(4):
            stage = nn.Sequential(
                *[Block(dim=dims[i], drop_path=dp_rates[cur + j]) for j in range(depths[i])]
            )
            self.stages.append(stage)
            cur += depths[i]

        # 4. Head (Classifier)
        self.norm = LayerNorm(dims[-1], eps=1e-6, data_format="channels_first") # final norm
        self.head = nn.Linear(dims[-1], num_classes)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
        
        # Global Average Pooling
        x = self.norm(x.mean([-2, -1], keepdim=True)) # (N, C, 1, 1)
        x = x.flatten(1)
        x = self.head(x)
        return x

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConvNeXt(num_classes=15).to(device)

# Model parameter checking

In [7]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

The number of your model parameters : 49464207
Parameter usage : 49.464207%


# Model training

In [8]:
print("GPU count:", torch.cuda.device_count())
print("Current device index:", torch.cuda.current_device())
print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

GPU count: 1
Current device index: 0
Current device name: NVIDIA GeForce RTX 4080 SUPER


In [9]:
import tqdm
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR, SequentialLR
from torch.cuda.amp import autocast

# 0. 벤치마크 켜기 (속도 향상)
torch.backends.cudnn.benchmark = True

scaler = torch.amp.GradScaler('cuda')

epochs = 15
save_path = "best_model.pth"
best_val_loss = float("inf")

criterion = nn.CrossEntropyLoss().to(device)
# AdamW와 높은 LR, Weight Decay 적용
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.05) 

# 스케줄러 설정 (Warmup + Cosine Annealing)
max_iterations = epochs * len(train_loader) # 총 스텝 수 계산 (15 * 281 = 4215)
warmup_steps = 1000 # 1000 스텝 동안 웜업 (약 3.5 에폭)
cosine_steps = max_iterations - warmup_steps # 코사인 Annealing이 적용될 남은 스텝 수

# 웜업 스케줄러 (LR을 0에서 4e-4까지 선형적으로 증가)
warmup_scheduler = LambdaLR(
    optimizer,
    lambda step: (step + 1) / warmup_steps if step < warmup_steps else 1.0,
)

# 코사인 Annealing 스케줄러
cosine_scheduler = CosineAnnealingLR(
    optimizer, T_max=cosine_steps
)

# 순차적 스케줄러 (Warmup 후 Cosine으로 전환)
scheduler = SequentialLR(
    optimizer, 
    schedulers=[warmup_scheduler, cosine_scheduler], 
    milestones=[warmup_steps]
)

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for x, y in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            output = model(x)
            loss = criterion(output, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # NOTE: 스케줄러는 스텝마다 호출해야 함
        scheduler.step()
        
        # 통계 (AMP로 계산된 output을 그대로 사용)
        train_loss_sum += loss.item() * y.size(0)
        
        # 예측값 계산 (여기는 그라디언트 필요 없으므로 detach 추천)
        preds = torch.argmax(output.detach(), dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # VALIDATION (그대로 유지)
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in tqdm.tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with autocast():
                output = model(x)
                val_loss_batch = criterion(output, y)

            val_loss_sum += val_loss_batch.item() * y.size(0)
            preds = torch.argmax(output, dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    # MODEL SAVE
    save_dict = {
        "epoch": epoch,
        "model_state_dict": (
            model.module.state_dict() if isinstance(model, nn.DataParallel)
            else model.state_dict()
        ),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_acc": val_acc,
    }

    torch.save(save_dict, f'epoch_{epoch}.pth')

    # BEST MODEL SPECIFICATION
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        save_dict = {
            "epoch": epoch,
            "model_state_dict": (
                model.module.state_dict() if isinstance(model, nn.DataParallel)
                else model.state_dict()
            ),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
        }

        torch.save(save_dict, save_path)
        print(f"Best model saved at epoch {epoch} (val_loss={val_loss:.4f})")

Epoch 0 [Train]:   0%|          | 0/562 [00:00<?, ?it/s]c:\Users\MAIN\anaconda3\envs\torch\lib\site-packages\torch\optim\lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Epoch 0 [Val]:   0%|          | 0/141 [00:00<?, ?it/s]C:\Users\MAIN\AppData\Local\Temp\ipykernel_18800\1587427411.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 0 [Val]: 100%|██████████| 141/141 [00:28<00:00,  5.00it/s]


Epoch 00 | Train Loss: 2.6420 | Train Acc: 0.1270 | Val Loss: 2.5671 | Val Acc: 0.1574
Best model saved at epoch 0 (val_loss=2.5671)


Epoch 1 [Train]:  78%|███████▊  | 437/562 [02:02<00:29,  4.19it/s]c:\Users\MAIN\anaconda3\envs\torch\lib\site-packages\torch\optim\lr_scheduler.py:209: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)
Epoch 1 [Val]: 100%|██████████| 141/141 [00:28<00:00,  4.98it/s]


Epoch 01 | Train Loss: 2.4007 | Train Acc: 0.2188 | Val Loss: 2.2654 | Val Acc: 0.2678
Best model saved at epoch 1 (val_loss=2.2654)


Epoch 2 [Val]: 100%|██████████| 141/141 [00:19<00:00,  7.37it/s]


...

Epoch 12 | Train Loss: 1.1561 | Train Acc: 0.6211 | Val Loss: 1.1712 | Val Acc: 0.6199
Best model saved at epoch 12 (val_loss=1.1712)


Epoch 13 [Val]: 100%|██████████| 141/141 [00:19<00:00,  7.39it/s]


Epoch 13 | Train Loss: 1.1260 | Train Acc: 0.6297 | Val Loss: 1.1723 | Val Acc: 0.6174


Epoch 14 [Val]: 100%|██████████| 141/141 [00:19<00:00,  7.29it/s]


Epoch 14 | Train Loss: 1.1083 | Train Acc: 0.6369 | Val Loss: 1.1639 | Val Acc: 0.6258
Best model saved at epoch 14 (val_loss=1.1639)


In [10]:
'''
# 모델 불러오기
checkpoint = torch.load("best_model.pth", map_location=device)

# 먼저 순수 모델을 만들고 로드
base_model = ConvNeXtBN(num_classes=15)
base_model.load_state_dict(checkpoint["model_state_dict"])

# 그 다음에 DataParallel로 감쌈
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
'''

'\n# 모델 불러오기\ncheckpoint = torch.load("best_model.pth", map_location=device)\n\n# 먼저 순수 모델을 만들고 로드\nbase_model = ConvNeXtBN(num_classes=15)\nbase_model.load_state_dict(checkpoint["model_state_dict"])\n\n# 그 다음에 DataParallel로 감쌈\nif torch.cuda.device_count() > 1:\n    model = torch.nn.DataParallel(base_model)\nelse:\n    model = base_model\n\nmodel = model.to(device)\n\noptimizer = torch.optim.Adam(model.parameters(), lr=0.001)\noptimizer.load_state_dict(checkpoint["optimizer_state_dict"])\n'

In [11]:
'''
# 모델 로드 검증
missing_keys, unexpected_keys = base_model.load_state_dict(
    checkpoint["model_state_dict"], strict=False
)

print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

# 파라미터 값 확인
with torch.no_grad():
    w = base_model.downsample_layers[0][0].weight

print("Sample weight stats:")
print("  mean:", w.mean().item())
print("  std :", w.std().item())
print("  min :", w.min().item())
print("  max :", w.max().item())

# forward 테스트
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(device)
    out = model(dummy)

print("Output shape:", out.shape)
print("Has NaN:", torch.isnan(out).any().item())
'''

'\n# 모델 로드 검증\nmissing_keys, unexpected_keys = base_model.load_state_dict(\n    checkpoint["model_state_dict"], strict=False\n)\n\nprint("Missing keys:", missing_keys)\nprint("Unexpected keys:", unexpected_keys)\n\n# 파라미터 값 확인\nwith torch.no_grad():\n    w = base_model.downsample_layers[0][0].weight\n\nprint("Sample weight stats:")\nprint("  mean:", w.mean().item())\nprint("  std :", w.std().item())\nprint("  min :", w.min().item())\nprint("  max :", w.max().item())\n\n# forward 테스트\nmodel.eval()\nwith torch.no_grad():\n    dummy = torch.randn(2, 3, 224, 224).to(device)\n    out = model(dummy)\n\nprint("Output shape:", out.shape)\nprint("Has NaN:", torch.isnan(out).any().item())\n'

In [12]:
'''
print("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))
print("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))
print("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))
'''

'\nprint("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))\nprint("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))\nprint("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))\n'

# Submit
Do not edit the submission code below.

In [13]:
submit = pd.read_csv('./cs441-assn3-data/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)

The number of your model parameters : 49464207
Parameter usage : 49.464207%


100%|██████████| 118/118 [00:23<00:00,  4.99it/s]
